# 4. Evaluate masks and candidate coordinates

Evaluate saved predictions against manual annotations without rerunning detection. Predictions and ground truth must remain separate. Only tobacco detection has quantitative historical evidence; these evaluators do not establish physical treatment success.

In [ ]:
from pathlib import Path
import sys
import os
import json
import tempfile

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks').is_dir() and (p / 'experiments/splits/items.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or a subdirectory.')
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import new_run, capture_environment
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'tobacco-notebook-mpl'))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'experiments/.ultralytics'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
RESULTS = ROOT / 'experiments/results'


## Configure independent evaluation

No previous kernel state is needed. Enable evaluation only after supplying manual annotations. Results go to a fresh run directory.

In [ ]:
RUN_EVALUATION = False
PREDICTIONS = ROOT / 'experiments/reproductions/mask-reproduction-01/masks'
GT_MASKS = ROOT / 'data/mask_validation/gt_masks'
GT_CENTERS = ROOT / 'data/mask_validation/centers'
RUN_NAME = 'evaluation-01'
THRESHOLD = 127
VARIANTS = ['hsv_only', 'crop_exclusion_only', 'hsv_crop_exclusion', 'final_contour_filtering']
evaluation_run = None


In [ ]:
import csv
import math
MASK_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


## Load paired binary masks

Use 8-bit masks with **255 for weed and 0 for background**. The evaluator uses `value > 127`; masks encoded as 0/1 must be converted before evaluation. Shapes must match the original image, and filenames must share stems. Never place generated predictions in the ground-truth directory.

In [ ]:
def read_mask(path: Path, threshold: int):
    import cv2
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read mask: {path}")
    return image > threshold

def find_masks(directory: Path) -> dict[str, Path]:
    masks = {}
    for path in sorted(directory.iterdir()):
        if path.suffix.lower() in MASK_EXTENSIONS:
            masks[path.stem] = path
    return masks


## Count pixels and compute overlap

IoU = TP/(TP+FP+FN); Dice = 2TP/(2TP+FP+FN); precision = TP/(TP+FP); recall = TP/(TP+FN). A zero denominator yields 1, preserving the historical evaluator convention.

In [ ]:
def compute_metrics(gt, pred):
    import numpy as np
    if gt.shape != pred.shape:
        raise ValueError(f"Mask shape mismatch: {gt.shape} vs {pred.shape}")
    tp = int(np.logical_and(gt, pred).sum())
    fp = int(np.logical_and(~gt, pred).sum())
    fn = int(np.logical_and(gt, ~pred).sum())
    tn = int(np.logical_and(~gt, ~pred).sum())
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 1.0
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    dice = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 1.0
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "dice": dice,
    }


## Evaluate a mask variant

Match by image stem. Reject incomplete predictions. Preserve six-decimal per-image rounding and the mean of rounded per-image values, rather than pooled pixel counts.

In [ ]:
def evaluate_masks(ground_truth, predicted, output, threshold=127):
    gt_masks = find_masks(ground_truth)
    pred_masks = find_masks(predicted)
    if not gt_masks:
        raise ValueError(f"No ground-truth masks found in {ground_truth}")
    if not pred_masks:
        raise ValueError(f"No predicted masks found in {predicted}")

    missing = sorted(set(gt_masks) - set(pred_masks))
    if missing:
        raise FileNotFoundError(f'Missing mask predictions: {missing}')
    rows = []
    for stem, gt_path in gt_masks.items():
        pred_path = pred_masks.get(stem)
        if pred_path is None:
            rows.append({"image": stem, "status": "missing_prediction"})
            continue
        metrics = compute_metrics(read_mask(gt_path, threshold), read_mask(pred_path, threshold))
        row = {"image": stem, "status": "ok"}
        row.update({key: f"{value:.6f}" if isinstance(value, float) else value for key, value in metrics.items()})
        rows.append(row)

    ok_rows = [row for row in rows if row["status"] == "ok"]
    if ok_rows:
        mean_row = {"image": "__mean__", "status": "ok"}
        for key in ["iou", "precision", "recall", "dice"]:
            mean_row[key] = f"{sum(float(row[key]) for row in ok_rows) / len(ok_rows):.6f}"
        rows.append(mean_row)

    output.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["image", "status", "tp", "fp", "fn", "tn", "iou", "precision", "recall", "dice"]
    with output.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote {output}")

    return rows


## Measure nearest-point distances

The current evaluator measures the distance from each ground-truth point to its nearest predicted point and reports hit rates within 10, 20, and 50 pixels. This is **not one-to-one matching**: one prediction can serve multiple ground-truth points, and extra predictions are not penalized. Therefore it does not measure false-target safety.

With ground-truth points but no predictions, distance is infinite and hit rates are zero. With no ground-truth points, hit rates are 1 by convention and mean distance is infinite. Inspect point counts alongside distances. Missing prediction files are rejected here rather than interpreted silently as empty predictions. Header-only `x,y` files represent valid empty predictions.

In [ ]:
def read_points(path: Path) -> list[tuple[float, float]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        return [(float(row["x"]), float(row["y"])) for row in reader if row.get("x") and row.get("y")]

def nearest_distances(predicted: list[tuple[float, float]], truth: list[tuple[float, float]]) -> list[float]:
    distances: list[float] = []
    for gt_x, gt_y in truth:
        if not predicted:
            distances.append(math.inf)
            continue
        distances.append(min(math.hypot(gt_x - pred_x, gt_y - pred_y) for pred_x, pred_y in predicted))
    return distances

def hit_rate(distances: list[float], threshold: float) -> float:
    if not distances:
        return 1.0
    return sum(distance <= threshold for distance in distances) / len(distances)


## Export coordinate measurements

Header-only x,y files are valid empty predictions. Missing files are errors. Additional predictions are not penalized and one prediction can match several truth points.

In [ ]:
def evaluate_coordinates(ground_truth, predicted, output):
    if not predicted.exists():
        raise ValueError(f"Missing predicted coordinate directory: {predicted}")

    truth_files = sorted(ground_truth.glob('*.csv'))
    if not truth_files:
        raise ValueError('No manual coordinate annotations found.')
    missing = [p.name for p in truth_files if not (predicted / p.name).is_file()]
    if missing:
        raise FileNotFoundError(f'Missing center predictions: {missing}')
    rows: list[dict[str, str]] = []
    for gt_path in sorted(ground_truth.glob("*.csv")):
        truth = read_points(gt_path)
        predicted = read_points(predicted / gt_path.name)
        distances = nearest_distances(predicted, truth)
        finite = [distance for distance in distances if math.isfinite(distance)]
        mean_error = sum(finite) / len(finite) if finite else math.inf
        rows.append(
            {
                "image": gt_path.stem,
                "predicted_points": str(len(predicted)),
                "ground_truth_points": str(len(truth)),
                "mean_nearest_error_px": f"{mean_error:.6f}" if math.isfinite(mean_error) else "inf",
                "hit_rate_10px": f"{hit_rate(distances, 10):.6f}",
                "hit_rate_20px": f"{hit_rate(distances, 20):.6f}",
                "hit_rate_50px": f"{hit_rate(distances, 50):.6f}",
            }
        )

    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "image",
                "predicted_points",
                "ground_truth_points",
                "mean_nearest_error_px",
                "hit_rate_10px",
                "hit_rate_20px",
                "hit_rate_50px",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {output}")

    return rows


## Run available evaluations

If neither annotation set exists, stop before creating a run. Missing annotation types are reported as unavailable, never as zero accuracy.

In [ ]:
if RUN_EVALUATION:
    has_masks = GT_MASKS.is_dir() and any(GT_MASKS.glob('*.png'))
    has_centers = GT_CENTERS.is_dir() and any(GT_CENTERS.glob('*.csv'))
    if not has_masks and not has_centers:
        raise FileNotFoundError('Supply manual masks or center CSVs before evaluation.')
    if not PREDICTIONS.is_dir():
        raise FileNotFoundError(PREDICTIONS)
    evaluation_run = new_run(ROOT, RUN_NAME)
    capture_environment(evaluation_run / 'environment')
    (evaluation_run / 'evaluation_config.json').write_text(json.dumps(dict(
        predictions=str(PREDICTIONS.resolve()), ground_truth_masks=str(GT_MASKS.resolve()),
        ground_truth_centers=str(GT_CENTERS.resolve()), threshold=THRESHOLD), indent=2))
    if has_masks:
        summaries = []
        for variant in VARIANTS:
            rows = evaluate_masks(GT_MASKS, PREDICTIONS / variant,
                                  evaluation_run / f'mask_{variant}.csv', THRESHOLD)
            mean = rows[-1]
            summaries.append(dict(variant=variant, images=len(rows)-1,
                **{f'mean_{key}': mean[key] for key in ['iou', 'precision', 'recall', 'dice']}))
        summary = pd.DataFrame(summaries)
        summary.to_csv(evaluation_run / 'mask_metrics.csv', index=False)
        display(summary)
    else:
        print('Mask accuracy unavailable: no manual masks supplied.')
    if has_centers:
        rows = evaluate_coordinates(GT_CENTERS, PREDICTIONS / 'centers',
                                    evaluation_run / 'coordinate_metrics.csv')
        display(pd.DataFrame(rows))
    else:
        print('Coordinate accuracy unavailable: no manual centers supplied.')
else:
    print('Evaluation disabled. No new quantitative evidence has been generated.')


## Interpret the outputs responsibly

Crop boxes can remove nearby weeds, and missed crops can become false weed candidates. Illumination affects fixed HSV thresholds. Contour centroids are geometric summaries rather than verified treatment points. Keep new prediction files, manual annotations, and historical evidence distinct.

The present article supports quantitative tobacco detection and qualitative mask/coordinate demonstrations. New manual annotations could support additional evaluation, but no notebook output alone validates physical laser treatment. Record dataset provenance, annotation policy, held-out image selection, settings, and hardware with any new results.